# Sequential control and memory : state of the art

**Level:** 3  
**Prerequisites:** Sequential control and memory level 1  
**Reading time:** 25 minutes

## Objectives

The learning objective is to explain the problem and its data, then inspect and, at higher levels, execute the released method on its matched task. Delayed observations, evolving job state and action consequences make software control sequential. The matrix distinguishes recurrent imitation, a myopic contextual bandit, masked PPO and bounded learned-model lookahead instead of grouping them under a generic ‘AI’ label.

## Setup

The versions below are pinned in `notebooks/requirements.txt` and are also recorded in this notebook's metadata.

```bash
pip install reflexmesh==0.1.2 numpy==2.4.6 matplotlib==3.11.2 torch==2.14.1 joblib==1.6.0 gymnasium==1.4.0 scikit-learn==1.9.1 stable-baselines3==2.9.2 sb3-contrib==2.9.2
```

## Data

The notebook reads committed product artifacts. Owner: Neuraxis. Repository: https://github.com/fsantibanezleal/CAOS_Neuraxis. License: Apache-2.0. The JSON is a first-party, frozen experiment artifact; no external benchmark data are redistributed in this notebook.

This notebook reads `data/derived/index.json` and the listed trace artifacts. The method run uses an isolated Reflexmesh case workspace. For learned methods, fetch the exact public release assets with `python -m reflexmesh.cli fetch-models --destination models/checkpoints --version v0.01.000`, then set `NEURAXIS_CHECKPOINTS` to that folder. The release manifest verifies asset hashes. Planner methods M09/M10 are read from frozen traces unless `NEURAXIS_NOTEBOOK_MODEL` explicitly names a configured local Ollama model.

Artifacts for this lesson:

- `data/derived/index.json`
- `data/derived/benchmark-adequacy.json`
- `data/derived/episodes/M05-C07-nominal-40000.json`
- `data/derived/episodes/M06-C07-nominal-40000.json`
- `data/derived/episodes/M07-C07-nominal-40000.json`
- `data/derived/episodes/M08-C07-nominal-40000.json`

## Method

Read the C07 cancellation slice, then inspect success by variant and the saved decision/effect sequence. The traces show which controller action actually ran and whether the independent outcome verifier accepted it. The notebook uses episode-level denominators and keeps seed and family structure visible.

Sequential RL optimizes \(G_t=\sum_{k=0}^{T-t}\gamma^k r_{t+k}\); a learned transition policy instead uses \(\hat s_{t+1}=f_\phi(s_t,a_t)\) to score a bounded search. These are package methods M07 and M08, not the same objective and not equivalent to a one-step bandit.

### Check the pinned analysis environment.

In [1]:
# Setup is performed outside the notebook; report the exact runtime pins.
import importlib.metadata
for distribution in ('reflexmesh', 'numpy', 'matplotlib', 'torch', 'joblib', 'gymnasium', 'scikit-learn', 'stable-baselines3', 'sb3-contrib'):
    print('{}=={}'.format(distribution, importlib.metadata.version(distribution)))

reflexmesh==0.1.2
numpy==2.4.6
matplotlib==3.11.2
torch==2.14.1
joblib==1.6.0
gymnasium==1.4.0
scikit-learn==1.9.1
stable-baselines3==2.9.2
sb3-contrib==2.9.2


### Verify the audit and each trace's bytes before interpreting outcomes.

In [2]:
# Read committed evidence and use the released API; never define another policy.
import hashlib, json
from pathlib import Path
ROOT = Path('../../../').resolve()
index_path = ROOT / 'data/derived/index.json'
index_bytes = index_path.read_bytes()
index = json.loads(index_bytes)
audit = json.loads((ROOT / 'data/derived/benchmark-adequacy.json').read_text(encoding='utf-8'))
assert hashlib.sha256(index_bytes).hexdigest() == audit['index_sha256']
selected_methods = ['M05', 'M06', 'M07', 'M08']
sample_case = 'C07'
seed = 40000
records = {}
for method_id in selected_methods:
    row = next(r for r in index['episodes'] if r['method_id'] == method_id and r['case_id'] == sample_case and r['variant'] == 'nominal' and r['seed'] == seed)
    trace_path = ROOT / 'data/derived' / row['path']
    trace_bytes = trace_path.read_bytes()
    assert hashlib.sha256(trace_bytes).hexdigest() == row['sha256']
    records[method_id] = json.loads(trace_bytes)
print('Index digest:', hashlib.sha256(index_bytes).hexdigest())
print('Evidence records:', len(records), '| case:', sample_case, '| seed:', seed)
print('Methods:', ', '.join(selected_methods))

Index digest: 1024a451fdc1f1aebc4e8d8d66e6bf1aca119f26ae058d3de9b84ff312d8d7a6
Evidence records: 4 | case: C07 | seed: 40000
Methods: M05, M06, M07, M08


### Inspect the original decisions and outcomes

In [3]:
# Summarize the recorded first action, authorized candidates, and independent oracle outcome.
for method_id, episode in records.items():
    first = episode['decisions'][0]
    allowed = [candidate['action'] for candidate in first['candidates'] if candidate['allowed']]
    print(method_id, '| action=', first['action'], '| reason=', first['reason'], '| allowed=', allowed)
    print('  success=', episode['metrics']['task_success'], '| steps=', episode['metrics']['steps'], '| violations=', episode['metrics']['violations'])

M05 | action= a-cancel | reason= recurrent_behavior_cloning | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M06 | action= a-cancel | reason= linucb_myopic | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M07 | action= a-cancel | reason= trained_nonrecurrent_maskable_ppo | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M08 | action= a-cancel | reason= learned_transition_bounded_search | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0


### Execute the released policies and compare the run

In [4]:
# Planner outcomes come from frozen traces; no LLM call is implicit.
import os
from pathlib import Path
from reflexmesh.environments import CaseSpec, SoftwareEnvironment
from reflexmesh.pipeline import make_policy
checkpoint_dir = Path(os.environ.get('NEURAXIS_CHECKPOINTS', str(ROOT / 'models/checkpoints')))
planner_model = os.environ.get('NEURAXIS_NOTEBOOK_MODEL')
reruns = {}
for method_id in selected_methods:
    if method_id in ('M09', 'M10') and not planner_model:
        print(method_id, 'not rerun: planner needs explicit NEURAXIS_NOTEBOOK_MODEL; frozen result remains above')
        continue
    policy = make_policy(method_id, checkpoint_dir)
    policy.reset('neuraxis-notebook-' + method_id)
    with SoftwareEnvironment(CaseSpec(sample_case, 'nominal', seed)) as env:
        while not env.terminal:
            _, outcome = env.step(policy.predict(env.observe()))
        success = env.verify()
    recorded = records[method_id]
    reruns[method_id] = {'success': success, 'recorded_success': bool(recorded['metrics']['task_success']), 'recorded_first_action': recorded['decisions'][0]['action']}
    print(method_id, reruns[method_id])

M05 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-cancel'}
M06 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-cancel'}
M07 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-cancel'}
M08 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-cancel'}


### Compare the full sequence, not only the first action

In [5]:
# Sequence equality is diagnostic; the independent verifier determines task success.
for method_id, episode in records.items():
    actions = [decision['action'] for decision in episode['decisions']]
    print(method_id, ' -> '.join(actions), '| terminal=', episode['status'], '| verified=', episode['metrics']['task_success'])

M05 a-cancel | terminal= succeeded | verified= 1
M06 a-cancel | terminal= succeeded | verified= 1
M07 a-cancel | terminal= succeeded | verified= 1
M08 a-cancel | terminal= succeeded | verified= 1


### Recompute per-variant outcomes from every trace in the case

In [6]:
# Read every seed for the selected case and keep each failed episode in the denominator.
from collections import defaultdict
groups = defaultdict(list)
for row in index['episodes']:
    if row['case_id'] not in (sample_case,) or row['method_id'] not in selected_methods:
        continue
    raw = (ROOT / 'data/derived' / row['path']).read_bytes()
    assert hashlib.sha256(raw).hexdigest() == row['sha256']
    episode = json.loads(raw)
    groups[(row['method_id'], row['variant'])].append(int(bool(episode['metrics']['task_success'])))
for (method_id, variant), values in sorted(groups.items()):
    print(method_id, variant, 'successes=', sum(values), 'episodes=', len(values), 'rate=', round(sum(values)/len(values), 3))

M05 boundary successes= 10 episodes= 10 rate= 1.0
M05 delayed_observation successes= 10 episodes= 10 rate= 1.0
M05 duplicate_event successes= 10 episodes= 10 rate= 1.0
M05 nominal successes= 10 episodes= 10 rate= 1.0
M05 stale_conflict successes= 10 episodes= 10 rate= 1.0
M05 unavailable_dependency successes= 10 episodes= 10 rate= 1.0
M06 boundary successes= 10 episodes= 10 rate= 1.0
M06 delayed_observation successes= 10 episodes= 10 rate= 1.0
M06 duplicate_event successes= 0 episodes= 10 rate= 0.0
M06 nominal successes= 10 episodes= 10 rate= 1.0
M06 stale_conflict successes= 0 episodes= 10 rate= 0.0
M06 unavailable_dependency successes= 0 episodes= 10 rate= 0.0
M07 boundary successes= 10 episodes= 10 rate= 1.0
M07 delayed_observation successes= 10 episodes= 10 rate= 1.0
M07 duplicate_event successes= 10 episodes= 10 rate= 1.0
M07 nominal successes= 10 episodes= 10 rate= 1.0
M07 stale_conflict successes= 10 episodes= 10 rate= 1.0
M07 unavailable_dependency successes= 10 episodes= 10 ra

### Plot the variant stress without pooling families

In [7]:
# Each point summarizes the seeds from one case variant, not the full benchmark.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
variants = sorted(set(key[1] for key in groups))
fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
for method_id in selected_methods:
    rates = [sum(groups[(method_id, variant)])/len(groups[(method_id, variant)]) if groups[(method_id, variant)] else float('nan') for variant in variants]
    ax.plot(variants, rates, marker='o', label=method_id)
ax.set_ylim(-0.05, 1.05)
ax.set_ylabel('Verified success rate')
ax.set_title(sample_case + ' by variant')
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=.25)
ax.legend()
display(fig)
plt.close(fig)

<Figure size 900x450 with 1 Axes>

## State of the art

GRU, PPO and model-based planning are mature method families with extensive prior art. The relevant evidence is their executed performance on the same controlled postconditions, not their algorithm names. The frozen matrix exposes the difficult cancellation family and preserves the weak methods rather than hiding them.

## Results

All 40 M12 failures are in C07, while several methods succeed on all 200 C07 episodes. The result identifies a concrete gap; aggregate success alone does not establish whether recurrence, action masking, model error or cancellation timing caused it.

No generated outputs are committed by this authoring step; `scripts/run_notebooks.py` executes and stores them from a fresh kernel.

## Summary

History and future consequences matter in principle, but complexity is not proof of useful memory. Read the trace and the verified task result together; inspect failure variants before proposing a new mechanism.

## References

- [GRU](https://arxiv.org/abs/1406.1078)
- [PPO](https://arxiv.org/abs/1707.06347)
- [LINUCB](https://arxiv.org/abs/1003.0146)